## Load libraries

In [ ]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.statespace.sarimax import SARIMAX

## Config

In [ ]:
# ---------------- CONFIG ----------------
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# tuning window: same as other models
TUNE_START_DATE = pd.Timestamp("2007-04-01")
TUNE_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV: 10 years train, 1 year val (in months)
ROLLING_TRAIN_WINDOW = 120   # 120 months
ROLLING_VAL_WINDOW   = 12    # 12 months

# SARIMAX hyperparameter grid
# (keep this small-ish or it will be slow)
param_grid = list(ParameterGrid({
    "order_p":  [0, 1, 2],
    "order_d":  [1],        # usually 1 for house prices
    "order_q":  [0, 1],
    "seasonal_P": [0, 1],
    "seasonal_D": [1],      # seasonal differencing (12)
    "seasonal_Q": [0, 1],
    "seasonal_period": [12]
}))

print(f"Number of SARIMAX configs: {len(param_grid)}")

# feature lists (same as before)
continuous_cols = [
    "AverageNeighbourPrice","local_I","area_km2","centroid_x","centroid_y",
    "CoL_distance_km","LA_FE","sdlt_perc_threshold","dwelling_stock",
    "population","ashe_weekly","base_rate","claimant_count_prop",
    "planning_decisions_per_1000","planning_granted_prop",
    "rail_station_entry_exit","GDP","CPIH",
]

categorical_cols = [
    "LMIQuadrant__2","LMIQuadrant__3","LMIQuadrant__4",
    "Region_East of England","Region_London","Region_North East",
    "Region_North West","Region_South East","Region_South West",
    "Region_West Midlands","Region_Yorkshire and The Humber"
]

base_feature_cols = continuous_cols + categorical_cols

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Metric functions

In [ ]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """
    MASE using seasonal naive (lag m) error on TRAIN period.
    y_train should be the *original* target in the same units.
    """
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

## Load data

In [ ]:
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

mask_tune = (df[TIME_COL] >= TUNE_START_DATE) & (df[TIME_COL] <= TUNE_END_DATE)
df = df.loc[mask_tune].copy()

# ensure one row per (Date, AreaCode)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)

print(f"Tuning period: {dates[0].date()} → {dates[-1].date()}")
print("Total months:", T_total)
print("Number of LAs:", N)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[base_feature_cols + [TARGET_COL]] = (
    df_panel[base_feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# last-resort fill
missing_total = df_panel[base_feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after ffill/bfill. Filling with column means.")
    col_means = df_panel[base_feature_cols + [TARGET_COL]].mean()
    df_panel[base_feature_cols + [TARGET_COL]] = df_panel[base_feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[base_feature_cols + [TARGET_COL]].isna().sum().sum())

## Rolling origin and folds

In [ ]:
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW

while True:
    train_end_idx = start_idx      # exclusive
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > T_total:
        break

    train_start_date = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end_date   = dates[train_end_idx - 1]
    val_start_date   = dates[val_start_idx]
    val_end_date     = dates[val_end_idx - 1]

    fold_specs.append((train_end_idx, val_start_idx, val_end_idx,
                       train_start_date, train_end_date,
                       val_start_date, val_end_date))

    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")
for i, (_, _, _, ts, te, vs, ve) in enumerate(fold_specs, start=1):
    print(f"  Fold {i}: Train {ts:%Y-%m}–{te:%Y-%m}, Val {vs:%Y-%m}–{ve:%Y-%m}")


## Tuning

In [ ]:

results = []

# Flattened original target for MASE baseline (over full tuning window;
# we will subset per fold’s train range inside the loop)
y_full_orig_flat = (
    df_panel[TARGET_COL]
    .values
)  # length T_total * N, but we’ll reshape as needed

for cfg_id, params in enumerate(param_grid, start=1):
    print(f"\n=== Config {cfg_id}/{len(param_grid)} ===")
    print(params)

    fold_mae_list   = []
    fold_rmse_list  = []
    fold_smape_list = []
    fold_mase_list  = []
    folds_used      = 0

    p  = params["order_p"]
    d  = params["order_d"]
    q  = params["order_q"]
    P  = params["seasonal_P"]
    D  = params["seasonal_D"]
    Q  = params["seasonal_Q"]
    s  = params["seasonal_period"]

    for fold_no, (train_end_idx, val_start_idx, val_end_idx,
                  train_start_date, train_end_date,
                  val_start_date, val_end_date) in enumerate(fold_specs, start=1):

        print(f"  Fold {fold_no}: Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
              f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}")

        # --- slice train/val by time ---
        train_dates = dates[:train_end_idx]
        val_dates   = dates[val_start_idx:val_end_idx]

        df_train = df_panel.loc[(train_dates, slice(None)), :].copy()
        df_val   = df_panel.loc[(val_dates,   slice(None)), :].copy()

        # ── leak-free scaling of exogenous features ──
        scaler = StandardScaler()
        # fit on TRAIN only, all LAs
        scaler.fit(df_train[continuous_cols])

        df_train_scaled = df_train.copy()
        df_val_scaled   = df_val.copy()

        df_train_scaled.loc[:, continuous_cols] = scaler.transform(df_train[continuous_cols])
        df_val_scaled.loc[:,   continuous_cols] = scaler.transform(df_val[continuous_cols])

        exog_cols = continuous_cols + categorical_cols  # numeric features for SARIMAX

        # For metrics, we’ll accumulate all (LA, time) in this fold
        y_true_fold = []
        y_pred_fold = []

        # y_train for MASE baseline: all TRAIN observations (all LAs)
        y_train_fold = df_train[TARGET_COL].values  # original prices

        for la in la_order:
            sub_train = df_train_scaled.xs(la, level=ENTITY_COL).sort_index()
            sub_val   = df_val_scaled.xs(la, level=ENTITY_COL).sort_index()

            # If some LAs are missing entirely in this fold, skip
            if sub_train[TARGET_COL].isna().all() or sub_val[TARGET_COL].isna().all():
                continue

            endog_train = sub_train[TARGET_COL].values.astype(float)
            exog_train  = sub_train[exog_cols].values.astype(float)
            exog_val    = sub_val[exog_cols].values.astype(float)
            n_val       = len(sub_val)

            # Basic safety check on series length
            if len(endog_train) < (p + P * s + 5):  # very rough minimum length
                continue

            try:
                model = SARIMAX(
                    endog=endog_train,
                    exog=exog_train,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                res = model.fit(disp=False)

                # Forecast for validation window
                y_forecast = res.forecast(steps=n_val, exog=exog_val)
                y_true     = sub_val[TARGET_COL].values.astype(float)

                # Collect for fold metrics
                y_true_fold.append(y_true)
                y_pred_fold.append(y_forecast.values)

            except Exception as e:
                # Convergence or numerical issues; skip this LA
                print(f"    LA {la}: SARIMAX failed ({e}). Skipping this LA in this fold.")
                continue

        if not y_true_fold:
            print("    ❌ No successful LA fits in this fold. Skipping fold.")
            continue

        y_true_fold = np.concatenate(y_true_fold, axis=0)
        y_pred_fold = np.concatenate(y_pred_fold, axis=0)

        fold_mae   = mae(y_true_fold, y_pred_fold)
        fold_rmse  = rmse(y_true_fold, y_pred_fold)
        fold_smape = smape(y_true_fold, y_pred_fold)
        fold_mase  = mase(y_true_fold, y_pred_fold, y_train_fold, m=12)

        print(f"    Fold {fold_no} MAE(£)={fold_mae:,.1f}, RMSE(£)={fold_rmse:,.1f}, "
              f"sMAPE={fold_smape:.3f}%, MASE={fold_mase:.3f}")

        fold_mae_list.append(fold_mae)
        fold_rmse_list.append(fold_rmse)
        fold_smape_list.append(fold_smape)
        fold_mase_list.append(fold_mase)
        folds_used += 1

    if folds_used == 0:
        print("  ❌ No valid folds for this config. Skipping.")
        continue

    cfg_result = {
        "model_type": "SARIMAX_panel",
        "order": (p, d, q),
        "seasonal_order": (P, D, Q, s),
        "folds_used": folds_used,

        "MAE_mean":   float(np.mean(fold_mae_list)),
        "MAE_std":    float(np.std(fold_mae_list)),
        "RMSE_mean":  float(np.mean(fold_rmse_list)),
        "RMSE_std":   float(np.std(fold_rmse_list)),
        "sMAPE_mean": float(np.mean(fold_smape_list)),
        "sMAPE_std":  float(np.std(fold_smape_list)),
        "MASE_mean":  float(np.mean(fold_mase_list)),
        "MASE_std":   float(np.std(fold_mase_list)),
    }
    results.append(cfg_result)




## Results

In [ ]:
results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("RMSE_mean").reset_index(drop=True)
    print("\n=== TOP SARIMAX CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===")
    print(results_df.head(10))
    results_df.to_csv("../../results/sarimax_panel_rollingcv_results.csv", index=False)
    print("\nSaved tuning results to ../../results/sarimax_panel_rollingcv_results.csv")
else:
    print("\nNo successful configs to report.")